> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

# AMD Nested Cross-Validation + Multiple Seeds — Reviewer v1

Notebook ini dibuat untuk menjawab komentar reviewer tentang:

- kemungkinan optimistic bias karena tuning di luar outer fold;
- kebutuhan nested cross-validation;
- dokumentasi lengkap grid search;
- penggunaan satu seed saja;
- ketidakstabilan ranking pada dataset kecil;
- kebutuhan immutable fold assignments dan fold-level predictions.

## Desain

```text
Frozen outer folds (5)
        ↓
Outer training partition
        ↓
Inner stratified CV (4 folds)
        ↓
Grid search:
C, gamma, class weight, SMOTE k, PCA dimension
        ↓
Select by macro-F1
        ↓
Refit on complete outer training partition
        ↓
Evaluate once on untouched outer validation fold
```

Proses tersebut diulang untuk lima random seeds dan lima frozen-feature
scenarios.


## Install dependencies

Jalankan hanya bila environment belum memiliki paket berikut.


In [ ]:
%pip install -q numpy pandas scikit-learn imbalanced-learn


## Nested CV pipeline

Sesuaikan bagian `CONFIG`, lalu jalankan sampai selesai.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE


# ============================================================
# 0. CONFIGURATION
# ============================================================

CONFIG = {
    # Frozen production release
    "DATASET_ROOT": Path(r"AMD_DATASET_594_PRODUCTION_v1"),

    # Feature cache produced by baseline reproduction notebook
    "FEATURE_CACHE_ROOT": Path(
        r"AMD_BASELINE_REPRODUCTION_v1/feature_cache"
    ),

    # Output must remain outside frozen dataset
    "OUTPUT_ROOT": Path(r"AMD_NESTEDCV_MULTISEED_v1"),

    # Fixed outer folds from the production release
    "OUTER_FOLD_FILE": Path(
        r"AMD_DATASET_594_PRODUCTION_v1/outer_fold_assignments.csv"
    ),

    # Multiple seeds requested by reviewer.
    # Five seeds are a practical minimum; increase to 10 if resources allow.
    "SEEDS": [42, 123, 2026, 3407, 7777],

    # Inner CV for hyperparameter selection inside each outer-training split.
    "INNER_FOLDS": 4,

    # Primary optimization metric due to class imbalance.
    "MODEL_SELECTION_METRIC": "macro_f1",

    # Scenarios requested for nested tuning.
    # Start with the five frozen-feature scenarios.
    "SCENARIOS": [
        "A_HANDCRAFTED_32D",
        "B_VIT_B16_768D",
        "C_FUSION_800D",
        "D_VIT_B16_PCA128",
        "E_FUSION_PCA128_PROPOSED",
    ],

    # Search space. Every candidate is evaluated strictly inside the
    # outer-training data.
    "SVM_GRID": {
        "C": [0.1, 1.0, 10.0, 100.0],
        "gamma": ["scale", 0.001, 0.01, 0.1],
        "class_weight": [None, "balanced"],
    },

    # PCA choices. For non-PCA scenarios use [None].
    "PCA_COMPONENTS_GRID": [64, 128, 256],

    # SMOTE settings.
    "SMOTE_K_NEIGHBORS_GRID": [3, 5],

    # Whether to use probability=True. Needed later for calibration,
    # but increases training time.
    "ENABLE_PROBABILITY": True,

    # Save every tested inner configuration.
    "SAVE_ALL_INNER_RESULTS": True,
}


# ============================================================
# 1. REPRODUCIBILITY AND UTILITIES
# ============================================================

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def ensure_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


CONFIG["OUTPUT_ROOT"].mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. LOAD PRODUCTION MANIFEST AND FIXED OUTER FOLDS
# ============================================================

def load_dataset_table() -> pd.DataFrame:
    release_manifest = (
        CONFIG["DATASET_ROOT"]
        / "provenance"
        / "release_manifest_594_sha256.csv"
    )
    ensure_exists(release_manifest, "Release manifest")
    ensure_exists(CONFIG["OUTER_FOLD_FILE"], "Outer-fold assignment file")

    manifest = pd.read_csv(
        release_manifest,
        dtype={"sample_id": str},
    )
    folds = pd.read_csv(
        CONFIG["OUTER_FOLD_FILE"],
        dtype={"sample_id": str},
    )

    manifest["sample_id"] = manifest["sample_id"].str.strip()
    folds["sample_id"] = folds["sample_id"].str.strip()

    if len(manifest) != 594:
        raise ValueError(f"Expected 594 manifest rows; found {len(manifest)}")
    if manifest["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample_id values in release manifest")
    if folds["sample_id"].duplicated().any():
        raise ValueError("Duplicate sample_id values in outer-fold file")

    if "outer_fold" in manifest.columns:
        manifest = manifest.drop(columns=["outer_fold"])

    df = manifest.merge(
        folds[["sample_id", "outer_fold"]],
        on="sample_id",
        how="left",
        validate="one_to_one",
    )

    if df["outer_fold"].isna().any():
        raise ValueError("Missing outer fold for one or more samples")

    df["outer_fold"] = df["outer_fold"].astype(int)
    if sorted(df["outer_fold"].unique().tolist()) != [1, 2, 3, 4, 5]:
        raise ValueError("Expected immutable outer folds 1..5")

    expected_counts = {"NORMAL": 287, "AMD_DRY": 139, "AMD_WET": 168}
    observed_counts = df["class_label"].value_counts().to_dict()
    if observed_counts != expected_counts:
        raise ValueError(
            f"Unexpected class counts: {observed_counts}; "
            f"expected {expected_counts}"
        )

    return df.sort_values("sample_id").reset_index(drop=True)


df = load_dataset_table()
print("Validated production table:", df.shape)
print(df["class_label"].value_counts())
print(df["outer_fold"].value_counts().sort_index())


# ============================================================
# 3. LOAD ALIGNED FEATURE MATRICES
# ============================================================

def load_aligned_feature_matrix(
    feature_filename: str,
    ids_filename: str,
    expected_dim: int,
) -> np.ndarray:
    feature_path = CONFIG["FEATURE_CACHE_ROOT"] / feature_filename
    ids_path = CONFIG["FEATURE_CACHE_ROOT"] / ids_filename

    ensure_exists(feature_path, feature_filename)
    ensure_exists(ids_path, ids_filename)

    X = np.load(feature_path)
    ids = pd.read_csv(ids_path, dtype={"sample_id": str})
    ids["sample_id"] = ids["sample_id"].str.strip()

    expected_ids = df["sample_id"].tolist()
    observed_ids = ids["sample_id"].tolist()

    if observed_ids != expected_ids:
        raise ValueError(
            f"Feature/sample alignment mismatch for {feature_filename}"
        )

    if X.shape != (594, expected_dim):
        raise ValueError(
            f"Unexpected shape for {feature_filename}: {X.shape}"
        )

    return X.astype(np.float32)


X_handcrafted = load_aligned_feature_matrix(
    "handcrafted_32d.npy",
    "handcrafted_sample_ids.csv",
    32,
)

X_vit = load_aligned_feature_matrix(
    "vit_b16_768d.npy",
    "vit_b16_sample_ids.csv",
    768,
)

X_fusion = np.concatenate(
    [X_handcrafted, X_vit],
    axis=1,
).astype(np.float32)

FEATURES = {
    "A_HANDCRAFTED_32D": {
        "X": X_handcrafted,
        "use_pca": False,
    },
    "B_VIT_B16_768D": {
        "X": X_vit,
        "use_pca": False,
    },
    "C_FUSION_800D": {
        "X": X_fusion,
        "use_pca": False,
    },
    "D_VIT_B16_PCA128": {
        "X": X_vit,
        "use_pca": True,
    },
    "E_FUSION_PCA128_PROPOSED": {
        "X": X_fusion,
        "use_pca": True,
    },
}

labels = sorted(df["class_label"].unique().tolist())
label_to_int = {label: i for i, label in enumerate(labels)}
int_to_label = {i: label for label, i in label_to_int.items()}
y = df["class_label"].map(label_to_int).to_numpy(dtype=int)

sample_ids = df["sample_id"].to_numpy()
source_datasets = df["source_dataset"].to_numpy()
outer_folds = df["outer_fold"].to_numpy()


# ============================================================
# 4. PIPELINE FIT/PREDICT
# ============================================================

def fit_transform_fold(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    *,
    seed: int,
    smote_k_neighbors: int,
    pca_components: int | None,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Leakage-safe fold preprocessing:
    SMOTE -> StandardScaler -> optional PCA.

    Every object is fitted only on the current training partition.
    """
    smote = SMOTE(
        random_state=seed,
        k_neighbors=smote_k_neighbors,
    )
    X_train_smote, y_train_smote = smote.fit_resample(
        X_train,
        y_train,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_smote)
    X_valid_scaled = scaler.transform(X_valid)

    meta = {
        "train_n_original": len(y_train),
        "train_n_after_smote": len(y_train_smote),
        "pca_components": None,
        "pca_explained_variance": None,
    }

    if pca_components is not None:
        effective_components = min(
            int(pca_components),
            X_train_scaled.shape[0],
            X_train_scaled.shape[1],
        )
        pca = PCA(
            n_components=effective_components,
            random_state=seed,
        )
        X_train_final = pca.fit_transform(X_train_scaled)
        X_valid_final = pca.transform(X_valid_scaled)
        meta["pca_components"] = effective_components
        meta["pca_explained_variance"] = float(
            np.sum(pca.explained_variance_ratio_)
        )
    else:
        X_train_final = X_train_scaled
        X_valid_final = X_valid_scaled

    return X_train_final, X_valid_final, y_train_smote, meta


def train_predict(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    *,
    seed: int,
    C: float,
    gamma,
    class_weight,
) -> tuple[np.ndarray, np.ndarray | None, SVC, float, float]:
    model = SVC(
        kernel="rbf",
        C=C,
        gamma=gamma,
        class_weight=class_weight,
        probability=CONFIG["ENABLE_PROBABILITY"],
        random_state=seed,
    )

    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started

    started = time.perf_counter()
    y_pred = model.predict(X_valid)
    if CONFIG["ENABLE_PROBABILITY"]:
        y_prob = model.predict_proba(X_valid)
    else:
        y_prob = None
    predict_seconds = time.perf_counter() - started

    return y_pred, y_prob, model, fit_seconds, predict_seconds


# ============================================================
# 5. INNER NESTED SEARCH
# ============================================================

def build_candidate_grid(use_pca: bool) -> list[dict]:
    base = list(ParameterGrid(CONFIG["SVM_GRID"]))
    candidates = []

    pca_values = (
        CONFIG["PCA_COMPONENTS_GRID"] if use_pca else [None]
    )

    for svm_params in base:
        for pca_components in pca_values:
            for smote_k in CONFIG["SMOTE_K_NEIGHBORS_GRID"]:
                candidate = dict(svm_params)
                candidate["pca_components"] = pca_components
                candidate["smote_k_neighbors"] = smote_k
                candidates.append(candidate)

    return candidates


def score_candidate_inner_cv(
    X_outer_train: np.ndarray,
    y_outer_train: np.ndarray,
    *,
    scenario: str,
    outer_fold: int,
    seed: int,
    candidate_id: int,
    params: dict,
) -> dict:
    inner_splitter = StratifiedKFold(
        n_splits=CONFIG["INNER_FOLDS"],
        shuffle=True,
        random_state=seed,
    )

    fold_scores = []

    for inner_fold, (train_idx, valid_idx) in enumerate(
        inner_splitter.split(
            np.zeros(len(y_outer_train)),
            y_outer_train,
        ),
        start=1,
    ):
        X_train_raw = X_outer_train[train_idx]
        X_valid_raw = X_outer_train[valid_idx]
        y_train = y_outer_train[train_idx]
        y_valid = y_outer_train[valid_idx]

        (
            X_train,
            X_valid,
            y_train_resampled,
            preprocessing_meta,
        ) = fit_transform_fold(
            X_train_raw,
            y_train,
            X_valid_raw,
            seed=seed + inner_fold,
            smote_k_neighbors=params["smote_k_neighbors"],
            pca_components=params["pca_components"],
        )

        y_pred, _, model, fit_seconds, predict_seconds = train_predict(
            X_train,
            y_train_resampled,
            X_valid,
            seed=seed + inner_fold,
            C=params["C"],
            gamma=params["gamma"],
            class_weight=params["class_weight"],
        )

        fold_scores.append({
            "inner_fold": inner_fold,
            "accuracy": accuracy_score(y_valid, y_pred),
            "macro_f1": f1_score(
                y_valid,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_valid,
                y_pred,
            ),
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "support_vectors": int(model.n_support_.sum()),
            **preprocessing_meta,
        })

    fold_df = pd.DataFrame(fold_scores)

    return {
        "scenario": scenario,
        "outer_fold": outer_fold,
        "seed": seed,
        "candidate_id": candidate_id,
        "C": params["C"],
        "gamma": params["gamma"],
        "class_weight": str(params["class_weight"]),
        "pca_components": params["pca_components"],
        "smote_k_neighbors": params["smote_k_neighbors"],
        "mean_inner_accuracy": float(fold_df["accuracy"].mean()),
        "sd_inner_accuracy": float(fold_df["accuracy"].std(ddof=1)),
        "mean_inner_macro_f1": float(fold_df["macro_f1"].mean()),
        "sd_inner_macro_f1": float(fold_df["macro_f1"].std(ddof=1)),
        "mean_inner_balanced_accuracy": float(
            fold_df["balanced_accuracy"].mean()
        ),
        "mean_inner_fit_seconds": float(
            fold_df["fit_seconds"].mean()
        ),
        "mean_inner_predict_seconds": float(
            fold_df["predict_seconds"].mean()
        ),
    }


def select_best_candidate(inner_results: pd.DataFrame) -> pd.Series:
    metric_col = {
        "macro_f1": "mean_inner_macro_f1",
        "accuracy": "mean_inner_accuracy",
        "balanced_accuracy": "mean_inner_balanced_accuracy",
    }[CONFIG["MODEL_SELECTION_METRIC"]]

    # Deterministic tie-breaking:
    # 1. highest selected metric
    # 2. highest balanced accuracy
    # 3. lower model complexity (smaller C and PCA)
    # 4. smaller candidate_id
    ranked = inner_results.copy()
    ranked["_pca_sort"] = ranked["pca_components"].fillna(10**9)

    ranked = ranked.sort_values(
        by=[
            metric_col,
            "mean_inner_balanced_accuracy",
            "C",
            "_pca_sort",
            "candidate_id",
        ],
        ascending=[False, False, True, True, True],
    )

    return ranked.iloc[0]


# ============================================================
# 6. FULL NESTED CV ACROSS SEEDS
# ============================================================

all_outer_metrics = []
all_predictions = []
all_inner_results = []
all_selected_configs = []

for seed in CONFIG["SEEDS"]:
    set_global_seed(seed)
    print(f"\n===== SEED {seed} =====")

    for scenario in CONFIG["SCENARIOS"]:
        X = FEATURES[scenario]["X"]
        use_pca = FEATURES[scenario]["use_pca"]
        candidate_grid = build_candidate_grid(use_pca)

        print(
            f"\nScenario={scenario} | candidates={len(candidate_grid)}"
        )

        for outer_fold in [1, 2, 3, 4, 5]:
            outer_valid_idx = np.where(outer_folds == outer_fold)[0]
            outer_train_idx = np.where(outer_folds != outer_fold)[0]

            if set(outer_train_idx).intersection(set(outer_valid_idx)):
                raise RuntimeError("Outer train/validation overlap detected")

            X_outer_train = X[outer_train_idx]
            y_outer_train = y[outer_train_idx]
            X_outer_valid = X[outer_valid_idx]
            y_outer_valid = y[outer_valid_idx]

            current_inner_rows = []

            for candidate_id, params in enumerate(candidate_grid, start=1):
                result = score_candidate_inner_cv(
                    X_outer_train,
                    y_outer_train,
                    scenario=scenario,
                    outer_fold=outer_fold,
                    seed=seed,
                    candidate_id=candidate_id,
                    params=params,
                )
                current_inner_rows.append(result)

            inner_df = pd.DataFrame(current_inner_rows)
            best = select_best_candidate(inner_df)

            if CONFIG["SAVE_ALL_INNER_RESULTS"]:
                all_inner_results.extend(current_inner_rows)

            selected = {
                "scenario": scenario,
                "seed": seed,
                "outer_fold": outer_fold,
                "candidate_id": int(best["candidate_id"]),
                "C": float(best["C"]),
                "gamma": best["gamma"],
                "class_weight": best["class_weight"],
                "pca_components": (
                    None
                    if pd.isna(best["pca_components"])
                    else int(best["pca_components"])
                ),
                "smote_k_neighbors": int(best["smote_k_neighbors"]),
                "selection_metric": CONFIG["MODEL_SELECTION_METRIC"],
                "selected_inner_macro_f1": float(
                    best["mean_inner_macro_f1"]
                ),
                "selected_inner_accuracy": float(
                    best["mean_inner_accuracy"]
                ),
                "selected_inner_balanced_accuracy": float(
                    best["mean_inner_balanced_accuracy"]
                ),
            }
            all_selected_configs.append(selected)

            (
                X_train_final,
                X_valid_final,
                y_train_resampled,
                preprocessing_meta,
            ) = fit_transform_fold(
                X_outer_train,
                y_outer_train,
                X_outer_valid,
                seed=seed + 1000 + outer_fold,
                smote_k_neighbors=selected["smote_k_neighbors"],
                pca_components=selected["pca_components"],
            )

            class_weight_value = (
                None
                if selected["class_weight"] == "None"
                else selected["class_weight"]
            )

            (
                y_pred,
                y_prob,
                model,
                fit_seconds,
                predict_seconds,
            ) = train_predict(
                X_train_final,
                y_train_resampled,
                X_valid_final,
                seed=seed + 1000 + outer_fold,
                C=selected["C"],
                gamma=selected["gamma"],
                class_weight=class_weight_value,
            )

            accuracy = accuracy_score(y_outer_valid, y_pred)
            macro_f1 = f1_score(
                y_outer_valid,
                y_pred,
                average="macro",
                zero_division=0,
            )
            balanced_acc = balanced_accuracy_score(
                y_outer_valid,
                y_pred,
            )

            report = classification_report(
                y_outer_valid,
                y_pred,
                labels=list(range(len(labels))),
                target_names=labels,
                output_dict=True,
                zero_division=0,
            )

            all_outer_metrics.append({
                "scenario": scenario,
                "seed": seed,
                "outer_fold": outer_fold,
                "outer_train_n": len(outer_train_idx),
                "outer_validation_n": len(outer_valid_idx),
                "accuracy": accuracy,
                "macro_f1": macro_f1,
                "balanced_accuracy": balanced_acc,
                "fit_seconds": fit_seconds,
                "predict_seconds": predict_seconds,
                "support_vectors": int(model.n_support_.sum()),
                "selected_C": selected["C"],
                "selected_gamma": selected["gamma"],
                "selected_class_weight": selected["class_weight"],
                "selected_pca_components": selected["pca_components"],
                "selected_smote_k_neighbors": selected["smote_k_neighbors"],
                "selected_inner_macro_f1": selected[
                    "selected_inner_macro_f1"
                ],
                "selected_inner_accuracy": selected[
                    "selected_inner_accuracy"
                ],
                "pca_explained_variance": preprocessing_meta[
                    "pca_explained_variance"
                ],
                **{
                    f"recall_{label}": report[label]["recall"]
                    for label in labels
                },
            })

            for local_idx, global_idx in enumerate(outer_valid_idx):
                pred_row = {
                    "scenario": scenario,
                    "seed": seed,
                    "outer_fold": outer_fold,
                    "sample_id": sample_ids[global_idx],
                    "source_dataset": source_datasets[global_idx],
                    "true_label": int_to_label[
                        int(y_outer_valid[local_idx])
                    ],
                    "predicted_label": int_to_label[
                        int(y_pred[local_idx])
                    ],
                    "correct": int(
                        y_outer_valid[local_idx]
                        == y_pred[local_idx]
                    ),
                }
                if y_prob is not None:
                    for class_index, class_name in enumerate(labels):
                        pred_row[f"prob_{class_name}"] = float(
                            y_prob[local_idx, class_index]
                        )
                all_predictions.append(pred_row)

            print(
                f"seed={seed} | scenario={scenario} | "
                f"outer_fold={outer_fold} | "
                f"acc={accuracy:.4f} | macroF1={macro_f1:.4f} | "
                f"C={selected['C']} | gamma={selected['gamma']} | "
                f"pca={selected['pca_components']} | "
                f"smote_k={selected['smote_k_neighbors']}"
            )


# ============================================================
# 7. VALIDATE AND SAVE RAW OUTPUTS
# ============================================================

outer_metrics_df = pd.DataFrame(all_outer_metrics)
predictions_df = pd.DataFrame(all_predictions)
selected_configs_df = pd.DataFrame(all_selected_configs)

expected_prediction_rows = (
    len(CONFIG["SEEDS"])
    * len(CONFIG["SCENARIOS"])
    * 594
)

if len(predictions_df) != expected_prediction_rows:
    raise RuntimeError(
        f"Expected {expected_prediction_rows} predictions; "
        f"found {len(predictions_df)}"
    )

dup_keys = [
    "scenario",
    "seed",
    "sample_id",
]
if predictions_df.duplicated(dup_keys).any():
    raise RuntimeError(
        "Duplicate out-of-fold prediction detected for "
        "(scenario, seed, sample_id)"
    )

coverage = (
    predictions_df.groupby(["scenario", "seed"])["sample_id"]
    .nunique()
)
if not (coverage == 594).all():
    raise RuntimeError(
        "Not every scenario/seed contains exactly 594 sample predictions"
    )

outer_metrics_df.to_csv(
    CONFIG["OUTPUT_ROOT"] / "nested_outer_fold_metrics.csv",
    index=False,
)

predictions_df.to_csv(
    CONFIG["OUTPUT_ROOT"] / "nested_sample_level_predictions.csv",
    index=False,
)

selected_configs_df.to_csv(
    CONFIG["OUTPUT_ROOT"] / "selected_hyperparameters_by_seed_fold.csv",
    index=False,
)

if CONFIG["SAVE_ALL_INNER_RESULTS"]:
    pd.DataFrame(all_inner_results).to_csv(
        CONFIG["OUTPUT_ROOT"] / "all_inner_grid_search_results.csv",
        index=False,
    )


# ============================================================
# 8. MULTI-SEED SUMMARIES
# ============================================================

seed_level_rows = []

for (scenario, seed), group in predictions_df.groupby(
    ["scenario", "seed"],
    sort=False,
):
    y_true_seed = group["true_label"].to_numpy()
    y_pred_seed = group["predicted_label"].to_numpy()

    seed_level_rows.append({
        "scenario": scenario,
        "seed": seed,
        "n": len(group),
        "pooled_accuracy": accuracy_score(
            y_true_seed,
            y_pred_seed,
        ),
        "pooled_macro_f1": f1_score(
            y_true_seed,
            y_pred_seed,
            average="macro",
            zero_division=0,
        ),
        "pooled_balanced_accuracy": balanced_accuracy_score(
            y_true_seed,
            y_pred_seed,
        ),
    })

seed_level_df = pd.DataFrame(seed_level_rows)
seed_level_df.to_csv(
    CONFIG["OUTPUT_ROOT"] / "seed_level_pooled_metrics.csv",
    index=False,
)

multi_seed_summary = (
    seed_level_df.groupby("scenario", as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        mean_accuracy_across_seeds=("pooled_accuracy", "mean"),
        sd_accuracy_across_seeds=("pooled_accuracy", "std"),
        min_accuracy_across_seeds=("pooled_accuracy", "min"),
        max_accuracy_across_seeds=("pooled_accuracy", "max"),
        mean_macro_f1_across_seeds=("pooled_macro_f1", "mean"),
        sd_macro_f1_across_seeds=("pooled_macro_f1", "std"),
        min_macro_f1_across_seeds=("pooled_macro_f1", "min"),
        max_macro_f1_across_seeds=("pooled_macro_f1", "max"),
        mean_balanced_accuracy_across_seeds=(
            "pooled_balanced_accuracy",
            "mean",
        ),
        sd_balanced_accuracy_across_seeds=(
            "pooled_balanced_accuracy",
            "std",
        ),
    )
)

multi_seed_summary.to_csv(
    CONFIG["OUTPUT_ROOT"] / "multi_seed_summary.csv",
    index=False,
)


# ============================================================
# 9. HYPERPARAMETER SELECTION FREQUENCY
# ============================================================

selection_frequency = (
    selected_configs_df.groupby(
        [
            "scenario",
            "C",
            "gamma",
            "class_weight",
            "pca_components",
            "smote_k_neighbors",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="selection_count")
    .sort_values(
        ["scenario", "selection_count"],
        ascending=[True, False],
    )
)

selection_frequency.to_csv(
    CONFIG["OUTPUT_ROOT"] / "hyperparameter_selection_frequency.csv",
    index=False,
)


# ============================================================
# 10. SOURCE- AND CLASS-WISE MULTI-SEED PERFORMANCE
# ============================================================

source_rows = []

for (scenario, seed, source), group in predictions_df.groupby(
    ["scenario", "seed", "source_dataset"],
    sort=False,
):
    source_rows.append({
        "scenario": scenario,
        "seed": seed,
        "source_dataset": source,
        "n": len(group),
        "accuracy": accuracy_score(
            group["true_label"],
            group["predicted_label"],
        ),
        "macro_f1_available_classes": f1_score(
            group["true_label"],
            group["predicted_label"],
            average="macro",
            zero_division=0,
        ),
    })

pd.DataFrame(source_rows).to_csv(
    CONFIG["OUTPUT_ROOT"] / "source_wise_multi_seed_metrics.csv",
    index=False,
)

class_rows = []

for (scenario, seed, class_name), group in predictions_df.groupby(
    ["scenario", "seed", "true_label"],
    sort=False,
):
    class_rows.append({
        "scenario": scenario,
        "seed": seed,
        "class_label": class_name,
        "support": len(group),
        "recall": float(group["correct"].mean()),
    })

pd.DataFrame(class_rows).to_csv(
    CONFIG["OUTPUT_ROOT"] / "class_wise_multi_seed_recall.csv",
    index=False,
)


# ============================================================
# 11. MULTI-SEED PAIRED CORRECTNESS MATRIX
# ============================================================

# One file per seed so later McNemar/Cochran analyses remain properly paired.
paired_dir = CONFIG["OUTPUT_ROOT"] / "paired_matrices_by_seed"
paired_dir.mkdir(parents=True, exist_ok=True)

for seed in CONFIG["SEEDS"]:
    seed_df = predictions_df[predictions_df["seed"] == seed].copy()

    base = (
        seed_df[
            [
                "sample_id",
                "source_dataset",
                "true_label",
                "outer_fold",
            ]
        ]
        .drop_duplicates()
        .sort_values("sample_id")
    )

    if len(base) != 594:
        raise RuntimeError(
            f"Seed {seed}: base paired matrix does not contain 594 samples"
        )

    paired = base.copy()

    for scenario in CONFIG["SCENARIOS"]:
        current = seed_df[
            seed_df["scenario"] == scenario
        ][
            [
                "sample_id",
                "predicted_label",
                "correct",
            ]
        ].rename(
            columns={
                "predicted_label": f"pred_{scenario}",
                "correct": f"correct_{scenario}",
            }
        )

        paired = paired.merge(
            current,
            on="sample_id",
            how="left",
            validate="one_to_one",
        )

    paired.to_csv(
        paired_dir / f"paired_prediction_matrix_seed_{seed}.csv",
        index=False,
    )


# ============================================================
# 12. RUN METADATA AND AUDIT REPORT
# ============================================================

metadata = {
    "analysis_name": "AMD Nested CV Multi-Seed Reviewer v1",
    "dataset_root": str(CONFIG["DATASET_ROOT"]),
    "feature_cache_root": str(CONFIG["FEATURE_CACHE_ROOT"]),
    "n_samples": 594,
    "outer_folds": [1, 2, 3, 4, 5],
    "seeds": CONFIG["SEEDS"],
    "inner_folds": CONFIG["INNER_FOLDS"],
    "model_selection_metric": CONFIG["MODEL_SELECTION_METRIC"],
    "scenarios": CONFIG["SCENARIOS"],
    "svm_grid": CONFIG["SVM_GRID"],
    "pca_components_grid": CONFIG["PCA_COMPONENTS_GRID"],
    "smote_k_neighbors_grid": CONFIG["SMOTE_K_NEIGHBORS_GRID"],
    "probability_enabled": CONFIG["ENABLE_PROBABILITY"],
    "outer_fold_sha256": sha256_file(CONFIG["OUTER_FOLD_FILE"]),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "notes": [
        "Outer folds are immutable and never used during hyperparameter selection.",
        "All hyperparameter selection occurs inside the outer-training partition.",
        "SMOTE, StandardScaler, and PCA are fitted separately inside each inner fold.",
        "The final outer model is refitted only after the best configuration is selected.",
        "Macro-F1 is the primary model-selection criterion.",
        "All 594 samples receive one out-of-fold prediction per scenario and seed.",
        "Multiple seeds quantify ranking and metric stability.",
    ],
}

(CONFIG["OUTPUT_ROOT"] / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

report = """# Nested Cross-Validation and Multi-Seed Audit

## Purpose

This package addresses reviewer concerns about optimistic bias, incomplete
hyperparameter documentation, reliance on a single five-fold realization, and
lack of fold-level predictions.

## Design

- Five immutable outer folds from the frozen production release.
- Four-fold inner stratified cross-validation inside each outer-training set.
- Macro-F1 as the primary hyperparameter-selection criterion.
- Grid search over SVM C, gamma, class weighting, SMOTE neighbors, and PCA
  dimension where applicable.
- Five independent random seeds.
- Sample-level out-of-fold predictions for every scenario and seed.

## Leakage controls

- The outer validation fold is never used for hyperparameter selection.
- SMOTE is fitted inside the current training partition only.
- StandardScaler is fitted inside the current training partition only.
- PCA is fitted inside the current training partition only.
- Final outer-fold performance is evaluated once after configuration selection.

## Interpretation

The primary revised estimate should be based on the multi-seed nested-CV
summary rather than the earlier single-seed fixed-parameter result.
"""

(CONFIG["OUTPUT_ROOT"] / "AUDIT_REPORT.md").write_text(
    report,
    encoding="utf-8",
)

print("\n=== NESTED CV MULTI-SEED COMPLETED ===")
print(multi_seed_summary.to_string(index=False))
print("\nOutputs:", CONFIG["OUTPUT_ROOT"].resolve())
